In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Model 1: Unconstrained (Overfit) ──────────────────────
dt_overfit = DecisionTreeClassifier(random_state=42)
dt_overfit.fit(X_train, y_train)

train_acc_overfit = accuracy_score(y_train, dt_overfit.predict(X_train))
test_acc_overfit  = accuracy_score(y_test,  dt_overfit.predict(X_test))

# ── Model 2: Pre-Pruned (Controlled) ──────────────────────
dt_pruned = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42
)
dt_pruned.fit(X_train, y_train)

train_acc_pruned = accuracy_score(y_train, dt_pruned.predict(X_train))
test_acc_pruned  = accuracy_score(y_test,  dt_pruned.predict(X_test))

# ── Side-by-side comparison ───────────────────────────────
print(f"{'':20} {'OVERFIT':>10} {'PRUNED':>10}")
print(f"{'─'*40}")
print(f"{'Train Accuracy':20} {train_acc_overfit:>10.3f} {train_acc_pruned:>10.3f}")
print(f"{'Test Accuracy':20} {test_acc_overfit:>10.3f} {test_acc_pruned:>10.3f}")
print(f"{'Tree Depth':20} {dt_overfit.get_depth():>10} {dt_pruned.get_depth():>10}")
print(f"{'Leaf Nodes':20} {dt_overfit.get_n_leaves():>10} {dt_pruned.get_n_leaves():>10}")
print(f"{'─'*40}")
print(f"{'Variance (gap)':20} {train_acc_overfit-test_acc_overfit:>10.3f} {train_acc_pruned-test_acc_pruned:>10.3f}")

                        OVERFIT     PRUNED
────────────────────────────────────────
Train Accuracy            1.000      0.963
Test Accuracy             0.912      0.947
Tree Depth                    7          4
Leaf Nodes                   19          9
────────────────────────────────────────
Variance (gap)            0.088      0.015


In [2]:
# Find the optimal ccp_alpha automatically
dt_temp = DecisionTreeClassifier(random_state=42)
dt_temp.fit(X_train, y_train)

# Get the pruning path — all possible alpha values
path = dt_temp.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

# Train one tree per alpha, compare test accuracy
results = []
for alpha in ccp_alphas:
    dt = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    dt.fit(X_train, y_train)
    results.append({
        "alpha"      : round(alpha, 5),
        "train_acc"  : round(accuracy_score(y_train, dt.predict(X_train)), 3),
        "test_acc"   : round(accuracy_score(y_test,  dt.predict(X_test)),  3),
        "depth"      : dt.get_depth(),
        "leaves"     : dt.get_n_leaves()
    })

# Print the first 8 rows to see the pruning effect
import pandas as pd
print(pd.DataFrame(results).head(8).to_string(index=False))

  alpha  train_acc  test_acc  depth  leaves
0.00000      1.000     0.912      7      19
0.00218      0.996     0.921      6      15
0.00287      0.991     0.939      5      12
0.00293      0.989     0.939      5      11
0.00396      0.987     0.939      4      10
0.00425      0.985     0.939      4       9
0.00502      0.982     0.939      4       8
0.00527      0.978     0.930      4       7
